# Understanding Reducers in depth
Companion notebook for the **Understanding Reducers in depth** section of Chapter 4 (covers Overwrite, List, Dict and Numeric reducers).

By default, an update to a state key **overwrites** the old value. Reducers change that merge behaviour -- essential once nodes start running in parallel and touching the same key.

In [1]:
import operator
from typing import Annotated
from pydantic import BaseModel, Field


class ResearchState(BaseModel):
    """Research state demonstrating the four reducer types discussed in the chapter."""

    # Overwrite reducer (default): the newest value replaces the old one
    final_report: str = Field(default="", description="Overwritten on every update (default behaviour)")

    # List Reducer: concatenates lists from parallel node updates
    candidate_urls: Annotated[list[str], operator.add] = Field(
        default_factory=list,
        description="List of URLs identified as potentially relevant sources. Uses operator.add to merge results.",
    )

    # Dict Reducer: merges dictionaries from parallel node updates
    chat_history: Annotated[dict, operator.or_] = Field(
        default_factory=dict,
        description="Chat storage for conversation history. Uses operator.or_ to merge conversation data.",
    )

    # Numeric Reducer: adds numbers from parallel node updates instead of overwriting
    count: Annotated[int, operator.add] = Field(
        default=0,
        description="An integer field for counter, merged using operator.add.",
    )

A reducer is just a plain `(existing_value, new_value) -> merged_value` function. LangGraph calls it automatically whenever two nodes update the same key in the same superstep.

In [2]:
print("List reducer:", operator.add(["url_a"], ["url_b"]))
print("Dict reducer:", operator.or_({"turn_1": "hi"}, {"turn_2": "hello"}))
print("Numeric reducer:", operator.add(1, 1))

List reducer: ['url_a', 'url_b']
Dict reducer: {'turn_1': 'hi', 'turn_2': 'hello'}
Numeric reducer: 2


To see why this matters, run two nodes **in parallel** that both update `candidate_urls` and `count`. Without a reducer only one node's update would survive; with `operator.add` both are kept.

In [3]:
from langgraph.graph import StateGraph, START, END


def add_wikipedia_url(state: ResearchState) -> dict:
    return {"candidate_urls": ["https://en.wikipedia.org/wiki/LangGraph"], "count": 1}


def add_web_url(state: ResearchState) -> dict:
    return {"candidate_urls": ["https://example.com/langgraph"], "count": 1}


workflow = StateGraph(ResearchState)
workflow.add_node("add_wikipedia_url", add_wikipedia_url)
workflow.add_node("add_web_url", add_web_url)

workflow.add_edge(START, "add_wikipedia_url")
workflow.add_edge(START, "add_web_url")
workflow.add_edge("add_wikipedia_url", END)
workflow.add_edge("add_web_url", END)

graph = workflow.compile()
result = graph.invoke(ResearchState())

print("candidate_urls:", result["candidate_urls"])
print("count:", result["count"], "(both parallel updates were merged, not overwritten)")

candidate_urls: ['https://example.com/langgraph', 'https://en.wikipedia.org/wiki/LangGraph']
count: 2 (both parallel updates were merged, not overwritten)
